# Data

In [1]:
import json

with open('data/result.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [2]:
import pandas as pd

df = pd.DataFrame(data['messages'])

In [3]:
from utils import preprocess_df

df = preprocess_df(df)

Загружено сообщений для анализа: 32738


# Pipeline

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import umap

reducer = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

In [6]:
from sklearn.cluster import HDBSCAN

clusterer = HDBSCAN()

In [7]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=model,
    hdbscan_model=clusterer,
    # umap_model=reducer,
    verbose=True
)

topics, probs = topic_model.fit_transform(df['clean_text'].tolist())

2026-03-25 12:37:18,043 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-03-25 12:39:24,142 - BERTopic - Embedding - Completed ✓
2026-03-25 12:39:24,143 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-25 12:39:56,716 - BERTopic - Dimensionality - Completed ✓
2026-03-25 12:39:56,717 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-25 12:40:06,952 - BERTopic - Cluster - Completed ✓
2026-03-25 12:40:06,971 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-25 12:40:07,816 - BERTopic - Representation - Completed ✓


# Analysis

In [8]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,10160,-1_d0_sex_тебе_там,"[d0, sex, тебе, там, да, что, если, ты, он, мы]","[бля ребят играл и такая вот хуйня, было у ког..."
1,0,218,0_заебался_лежал_насрал_ксан,"[заебался, лежал, насрал, ксан, днище, менту, ...",[я забросил это когда с помощью ачивмент анлок...
2,1,193,1_она_ей_нее_тя,"[она, ей, нее, тя, ее, неё, игнорит, её, хмпф,...","[да она бля, А че она, Как она]"
3,2,188,2_спорю_всасываю_жалуюсь_лизинг,"[спорю, всасываю, жалуюсь, лизинг, прикола, ск...","[Не спорю, Да ниче епта ранг всасываю, Да ниче..."
4,3,145,3_бреетиииш_легендыч_вотафа_гейпромхоззз,"[бреетиииш, легендыч, вотафа, гейпромхоззз, ku...","[Легендыч, Бреетиииш, Бреетиииш]"
...,...,...,...,...,...
1413,1412,5,1412_расширение_регионе_развития_района,"[расширение, регионе, развития, района, област...","[место развития, Это дело в регионе, расширени..."
1414,1413,5,1413_вешу_52_каянгел_79,"[вешу, 52, каянгел, 79, 76, 78, ему, че, не, ]","[а каянгел в 52?, Я 76 вешу, Я 78-79 вешу]"
1415,1414,5,1414_ручку_продай_бортоломную_афимолле,"[ручку, продай, бортоломную, афимолле, подскаж...","[Где в афимолле ручку купить можно подскажите,..."
1416,1415,5,1415_браузера_работает_дискорд_заходит,"[браузера, работает, дискорд, заходит, дс, них...","[Хули у меня нихуя не работает, у меня нихуя н..."


In [9]:
topic_model.get_topic_info().describe()

,Topic,Count
count,1418.000000,1418.000000
mean,707.500000,23.087447
std,409.485653,269.937104
min,-1.000000,5.000000
25%,353.250000,7.000000
50%,707.500000,11.000000
75%,1061.750000,19.000000
max,1416.000000,10160.000000
